In [59]:
from prepocess import *

node2id = preprocess(file_path="data/syn_net.csv")

20 nodes in the link stream


/Users/acw721/Desktop/research/linkstream/prepocess.py:4: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  link_stream = pd.read_csv(file_path, delimiter=delimiter, names=['source', 'destination', 'timestamp'], index_col=False, skiprows=1)


In [60]:
import pandas as pd

link_stream = pd.read_csv('data/syn_netpcs.csv')

In [61]:
(link_stream.head(5))

,source,destination,timestamp,idx
0,0,7,9,0
1,0,6,17,1
2,0,11,25,2
3,0,2,28,3
4,1,18,35,4


In [40]:
import torch
import pandas as pd
import numpy as np
from typing import Dict, Tuple, Optional


class OfflineStateManager:
    def __init__(self, num_nodes: int, num_communities: int, device: str = "cpu"):
        self.num_nodes = num_nodes
        self.num_communities = num_communities
        self.device = device

        # -------- static stats (precomputed once) --------
        self.global_degree = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        self.node_lifespans = torch.ones(num_nodes, device=device, dtype=torch.float32)
        self.total_duration = 1.0
        self.m = 0.0

        # -------- time index for delta_t --------
        self.node_time_pos = torch.zeros(num_nodes, dtype=torch.long, device=device)
        self.node_time_indptr: Optional[torch.Tensor] = None
        self.node_time_values: Optional[torch.Tensor] = None

        # -------- dynamic buffers (curr / old) --------
        self.A_curr: Optional[torch.Tensor] = None
        self.S_curr: Optional[torch.Tensor] = None

        self.A_old: Optional[torch.Tensor] = None 
        self.S_old: Optional[torch.Tensor] = None

        self.p_src_old_all: Optional[torch.Tensor] = None
        self.p_dst_old_all: Optional[torch.Tensor] = None
        self.num_instances_cached: int = 0
        # -------- other dynamic state --------
        self.p_prev: Optional[torch.Tensor] = None

        # 数值稳定用的小 eps
        self.eps = 1e-12

    # ---------------------- one-time preprocessing ----------------------
    def pre_scan_data(self, link_stream: pd.DataFrame):
        self.m = float(len(link_stream))
        print(f"Total links: {int(self.m)}")

        # ---- global_degree k_u ----
        all_nodes = pd.concat([link_stream["source"], link_stream["destination"]], ignore_index=True)
        all_nodes_t = torch.tensor(all_nodes.values, dtype=torch.long, device=self.device)
        self.global_degree = torch.bincount(all_nodes_t, minlength=self.num_nodes).float()

        # ---- total duration ----
        t_max = link_stream["timestamp"].max()
        t_min = link_stream["timestamp"].min()
        total_duration = t_max - t_min
        self.total_duration = float(total_duration) if total_duration > 0 else 1.0
        print(f"T_duration = {total_duration}")

        # ---- node lifespans (optional) ----
        sources = link_stream[["source", "timestamp"]].rename(columns={"source": "node"})
        destinations = link_stream[["destination", "timestamp"]].rename(columns={"destination": "node"})
        all_events = pd.concat([sources, destinations], ignore_index=True)

        node_stats = all_events.groupby("node")["timestamp"].agg(["min", "max"])
        epsilon = 1.0
        lifespans = (node_stats["max"] - node_stats["min"]).clip(lower=epsilon)

        self.node_lifespans.fill_(epsilon)
        active_ids = torch.tensor(node_stats.index.values, device=self.device, dtype=torch.long)
        active_vals = torch.tensor(lifespans.values, device=self.device, dtype=torch.float32)
        self.node_lifespans[active_ids] = active_vals
        print(f"Max Lifespan: {self.node_lifespans.max().item()}")

        # ---- build node_time_values: timestamps sorted by (node, timestamp) ----
        all_events_sorted = all_events.sort_values(["node", "timestamp"], kind="mergesort")
        nodes_np = all_events_sorted["node"].to_numpy(dtype=np.int64, copy=False)
        times_np = all_events_sorted["timestamp"].to_numpy(copy=False)

        if np.issubdtype(times_np.dtype, np.integer):
            if times_np.min() >= np.iinfo(np.int32).min and times_np.max() <= np.iinfo(np.int32).max:
                times_np = times_np.astype(np.int32, copy=False)
            else:
                times_np = times_np.astype(np.int64, copy=False)
        else:
            times_np = times_np.astype(np.float32, copy=False)

        counts = np.bincount(nodes_np, minlength=self.num_nodes).astype(np.int64)
        indptr = np.empty(self.num_nodes + 1, dtype=np.int64)
        indptr[0] = 0
        np.cumsum(counts, out=indptr[1:])

        self.node_time_indptr = torch.from_numpy(indptr).to(self.device)
        self.node_time_values = torch.from_numpy(times_np).to(self.device)

        print(f"Total node appearances stored: {self.node_time_values.numel()} (expected ~ {2*int(self.m)})")

        # epoch0 init
        self.reset_time_pos()
        self.reset_p_prev_uniform()
        self.init_old_uniform_prior()

    # ---------------------- epoch / pass control ----------------------
    def reset_time_pos(self):
        """Reset per-node appearance pointer (for delta_t)."""
        self.node_time_pos.zero_()

    def reset_p_prev_uniform(self):
        """Reset p_prev to uniform 1/K."""
        N, K = self.num_nodes, self.num_communities
        self.p_prev = torch.full((N, K), 1.0 / float(K), device=self.device, dtype=torch.float32)

    def reset_curr_from_zero(self):

        N, K = self.num_nodes, self.num_communities
        self.A_curr = torch.zeros((N, K), device=self.device, dtype=torch.float32)
        self.S_curr = torch.zeros((K,), device=self.device, dtype=torch.float32)

    def init_old_uniform_prior(self):
        N, K = self.num_nodes, self.num_communities
        Tu = self.node_lifespans.clamp_min(1.0)
        self.A_old = (Tu[:, None] / float(K)).expand(N,K).to(device=self.device, dtype=torch.float32).clone()
        self.S_old = (self.global_degree[:, None] * torch.sqrt(self.A_old)).sum(dim=0)

    @torch.no_grad()
    def finalize_curr(self):

        if self.A_curr is None:
            raise RuntimeError("A_curr is None. Call reset_curr_from_zero() before accumulating.")
        self.S_curr = (self.global_degree[:, None] * torch.sqrt(self.A_curr.clamp_min(self.eps))).sum(dim=0)


    @torch.no_grad()
    def promote_curr_to_old(self, copy_A: bool = True):
        if self.S_curr is None:
            raise RuntimeError("S_curr is None. Call finalize_curr() before promote_curr_to_old().")
        if copy_A:
            if self.A_curr is None:
                raise RuntimeError("A_curr is None. Cannot promote A.")
            self.A_old = self.A_curr.clone()
        self.S_old = self.S_curr.clone()

    # ---------------------- delta_t computation ----------------------
    def get_delta_for_batch(
        self,
        src: torch.Tensor,  # [B] long
        dst: torch.Tensor,  # [B] long
    ) -> Tuple[torch.Tensor, torch.Tensor, Dict[int, int]]:
        """
        Compute delta_src/delta_dst for each occurrence in this batch.
        Uses node_time_pos + temp_pos to handle repeated nodes within batch.

        Returns:
          delta_src, delta_dst: [B] float32
          occ_counts: {u: count_in_batch} for advancing node_time_pos
        """
        assert self.node_time_indptr is not None and self.node_time_values is not None, \
            "Call pre_scan_data() first to build node_time_indptr/node_time_values."

        assert src.dtype == torch.long and dst.dtype == torch.long
        B = src.numel()
        device = self.device

        delta_src = torch.zeros(B, device=device, dtype=torch.float32)
        delta_dst = torch.zeros(B, device=device, dtype=torch.float32)

        temp_pos: Dict[int, int] = {}
        occ_counts: Dict[int, int] = {}

        def _next_delta(u: int) -> float:
            base_pos = int(self.node_time_pos[u].item())
            offset = temp_pos.get(u, 0)
            pos = base_pos + offset

            start = int(self.node_time_indptr[u].item())
            end = int(self.node_time_indptr[u + 1].item())
            idx = start + pos

            temp_pos[u] = offset + 1
            occ_counts[u] = occ_counts.get(u, 0) + 1

            # last appearance => fallback
            if idx + 1 >= end:
                return 1.0

            t1 = self.node_time_values[idx]
            t2 = self.node_time_values[idx + 1]
            dt = (t2 - t1).float()
            dt = torch.clamp(dt, min=1.0)  # avoid 0/negative
            return float(dt.item())

        for i in range(B):
            u = int(src[i].item())
            v = int(dst[i].item())
            delta_src[i] = _next_delta(u)
            delta_dst[i] = _next_delta(v)

        return delta_src, delta_dst, occ_counts

    # ---------------------- commit after each batch ----------------------
    @torch.no_grad()
    def commit_batch(
        self,
        delta_a_nodes: Dict[int, torch.Tensor],  # {u: [K]}
        last_p_nodes: Dict[int, torch.Tensor],   # {u: [K]}
        occ_counts: Dict[int, int],              # {u: count}
        update_A_curr: bool,
    ):
        if update_A_curr:
            if self.A_curr is None:
                raise RuntimeError("A_curr is None. Call reset_curr_from_zero() before accumulating A_curr.")
            for u, dA in delta_a_nodes.items():
                self.A_curr[int(u)] += dA.detach()

        # update p_prev
        for u, p_last in last_p_nodes.items():
            self.p_prev[int(u)] = p_last.detach()

        # advance node_time_pos
        for u, cnt in occ_counts.items():
            self.node_time_pos[int(u)] += int(cnt)


    @torch.no_grad()
    def init_old_p_cache(
        self,
        num_instances: int,
        dtype: torch.dtype = torch.float16,
        pin_memory: bool = False,
        init_uniform: bool = True,
    ):
        K = self.num_communities
        self.num_instances_cached = int(num_instances)

        if init_uniform:
            val = 1.0 / float(K)
            p_src = torch.full((num_instances, K), val, dtype=dtype, device="cpu")
            p_dst = torch.full((num_instances, K), val, dtype=dtype, device="cpu")
        else:
            p_src = torch.zeros((num_instances, K), dtype=dtype, device="cpu")
            p_dst = torch.zeros((num_instances, K), dtype=dtype, device="cpu")

        pin_ok = pin_memory and torch.cuda.is_available()
        if pin_ok:
            p_src = p_src.pin_memory()
            p_dst = p_dst.pin_memory()

        self.p_src_old_all = p_src
        self.p_dst_old_all = p_dst



    @torch.no_grad()
    def write_old_p_batch(self, start_idx: int, end_idx: int, 
                        p_src: torch.Tensor, p_dst: torch.Tensor):
        if self.p_src_old_all is None or self.p_dst_old_all is None:
            raise RuntimeError("old p cache not initialized.")
        

        p_src_copy = p_src.clone().detach().cpu().to(dtype=self.p_src_old_all.dtype)
        p_dst_copy = p_dst.clone().detach().cpu().to(dtype=self.p_dst_old_all.dtype)
        
        self.p_src_old_all[start_idx:end_idx].copy_(p_src_copy)
        self.p_dst_old_all[start_idx:end_idx].copy_(p_dst_copy)


    def read_old_p_batch(
        self,
        start_idx: int,
        end_idx: int,
        device: torch.device,
        dtype: torch.dtype,
        non_blocking: bool = True,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Called in training pass:
        Fetch old p for this batch and move to GPU (or target device).
        Returns p_src_old, p_dst_old: [B,K] on `device` with `dtype`.
        """
        if self.p_src_old_all is None or self.p_dst_old_all is None:
            raise RuntimeError("old p cache not initialized. Call init_old_p_cache(num_instances) first.")

        p_src_old = self.p_src_old_all[start_idx:end_idx].to(device=device, dtype=dtype, non_blocking=non_blocking)
        p_dst_old = self.p_dst_old_all[start_idx:end_idx].to(device=device, dtype=dtype, non_blocking=non_blocking)
        return p_src_old, p_dst_old

In [96]:
device = 'cuda' if torch.cuda.is_available() else 'mps'
node_set = set(pd.concat([link_stream['source'], link_stream['destination']], ignore_index=True))
num_nodes = len(node_set)
print(f'num_nodes = {num_nodes} ')
num_comms = 2
state_mgr = OfflineStateManager(num_nodes, num_comms, device=device)
state_mgr.pre_scan_data(link_stream)
state_mgr.init_old_p_cache(len(link_stream), dtype=torch.float16, pin_memory=True, init_uniform=True)

num_nodes = 20 
Total links: 182
T_duration = 1087
Max Lifespan: 1061.0
Total node appearances stored: 364 (expected ~ 364)


In [97]:
import math
import logging
import time
import sys
import argparse
import numpy as np
import pickle
from pathlib import Path


NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_HEADS = 2
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.001
NODE_DIM = 16
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = 16
device = 'cuda' if torch.cuda.is_available() else 'mps'

prefix = 'max_lmod'
data='link_stream'

Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


In [99]:
from tgn.utils.my_data import get_data  
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

data = 'syn_netpcs'

node_feat, edge_feat, full_data = get_data(data)
max_idx = max(full_data.unique_nodes)

cannot find (./data/syn_netpcs.npy), use zero-vector (dim=16)...
cannot find node feature: ./data/syn_netpcs_node.npy), use zero vector(dim=16)...
The dataset has 182 interactions, involving 20 different nodes


In [44]:
'''
BATCH_SIZE = 200
num_instance = len(full_data.sources)

state_mgr.reset_time_pos()
for k in range(1, 2):
    
    start_idx = BATCH_SIZE * k
    end_idx = min(num_instance, BATCH_SIZE * (k + 1))

    sources_batch = full_data.sources[start_idx:end_idx]
    destinations_batch = full_data.destinations[start_idx:end_idx]
    timestamps_batch = full_data.timestamps[start_idx:end_idx]
    edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]
    
    src = torch.as_tensor(sources_batch, dtype=torch.long, device=device)
    dst = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)
    ts  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)
    print("src:", src)
    delta_src, delta_dst, occ_counts = state_mgr.get_delta_for_batch(src, dst)
    print("delta_src:", delta_src)
    print("occ count", occ_counts)
'''

'\nBATCH_SIZE = 200\nnum_instance = len(full_data.sources)\n\nstate_mgr.reset_time_pos()\nfor k in range(1, 2):\n\n    start_idx = BATCH_SIZE * k\n    end_idx = min(num_instance, BATCH_SIZE * (k + 1))\n\n    sources_batch = full_data.sources[start_idx:end_idx]\n    destinations_batch = full_data.destinations[start_idx:end_idx]\n    timestamps_batch = full_data.timestamps[start_idx:end_idx]\n    edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]\n\n    src = torch.as_tensor(sources_batch, dtype=torch.long, device=device)\n    dst = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)\n    ts  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)\n    print("src:", src)\n    delta_src, delta_dst, occ_counts = state_mgr.get_delta_for_batch(src, dst)\n    print("delta_src:", delta_src)\n    print("occ count", occ_counts)\n'

In [100]:
from tgn.utils.utils import EarlyStopMonitor, get_neighbor_finder, MLP

ngh_finder = get_neighbor_finder(full_data, uniform=True, max_node_idx=max_idx)

In [101]:
from tgn.utils.my_data import get_data, compute_time_statistics, compute_time_statistics_undirected
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

mean_time_shift, std_time_shift= compute_time_statistics_undirected(full_data.sources, 
                                                                    full_data.destinations, 
                                                                    full_data.timestamps)


In [102]:
from tgn.model.my_tgn import TGN
importlib.reload(sys.modules['tgn.model.my_tgn'])
from pathlib import Path


NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_HEADS = 2
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.001
NODE_DIM = 16
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = 16
num_communities = 2

device = 'cuda' if torch.cuda.is_available() else 'mps'
aggregator = 'last'
memory_update_at_end = False
embedding_module = 'graph_attention'
message_function = 'mlp'

prefix = 'syn_net'

tgn = TGN(
    neighbor_finder=ngh_finder,
    node_features=node_feat,
    edge_features=edge_feat,
    device=device,
    n_layers=NUM_LAYER,
    n_heads=NUM_HEADS,
    dropout=DROP_OUT,
    use_memory=USE_MEMORY,
    message_dimension=MESSAGE_DIM,
    memory_dimension=MEMORY_DIM,
    n_neighbors=NUM_NEIGHBORS,
    mean_time_shift=mean_time_shift,
    std_time_shift=std_time_shift,
    use_destination_embedding_in_message=True,
    use_source_embedding_in_message=True,
    memory_update_at_start=not memory_update_at_end,
    embedding_module_type=embedding_module,
    message_function=message_function,
    aggregator_type=aggregator, 
    
    num_communities=num_communities,
    dirichlet_alpha=5.0

).to(device)

optimizer = torch.optim.Adam(tgn.parameters(), lr=LEARNING_RATE)

In [103]:
import math 
import logging
import time


BATCH_SIZE = 200

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


num_instance = len(full_data.sources)
num_batches = math.ceil(len(full_data.sources) / BATCH_SIZE)
logger.debug(f'num_batches: {num_batches}')

DEBUG:root:num_batches: 1


In [104]:
def build_batch_updates(src, dst,
                        p_src_curr, p_dst_curr,
                        p_src_old,  p_dst_old,
                        delta_src, delta_dst):
    delta_a_nodes = {}
    last_p_nodes = {}

    def _acc(node_ids, p_curr, p_old, deltas):
        for i in range(node_ids.numel()):
            u = int(node_ids[i].item())
            dt = deltas[i]
            if dt.item() <= 0:
                continue
            dtv = dt.to(dtype=p_curr.dtype)
            inc = (p_curr[i] - p_old[i]) * dtv 
            delta_a_nodes[u] = delta_a_nodes.get(u, 0) + inc
            last_p_nodes[u] = p_curr[i]

    _acc(src, p_src_curr, p_src_old, delta_src)
    _acc(dst, p_dst_curr, p_dst_old, delta_dst)
    return delta_a_nodes, last_p_nodes

def build_batch_updates_abs(src, dst, p_src, p_dst, delta_src, delta_dst):
    delta_a_nodes = {}
    last_p_nodes = {}

    def _acc(node_ids, probs, deltas):
        for i in range(node_ids.numel()):
            u = int(node_ids[i].item())
            dt = deltas[i]
            if dt.item() <= 0:
                continue
            dtv = dt.to(dtype=probs.dtype)

            inc = probs[i] * dtv          # [K]
            delta_a_nodes[u] = delta_a_nodes.get(u, 0) + inc
            last_p_nodes[u] = probs[i]


    _acc(src, p_src, delta_src)
    _acc(dst, p_dst, delta_dst)

    return delta_a_nodes, last_p_nodes

In [105]:
def print_grads(tag: str):
    print(f"\n=== Batch {k+1} {tag} Gradients ===")
    for name, param in tgn.named_parameters():
        if param.grad is not None:
            g = param.grad
            print(f"{name}: norm={g.norm().item():.6f}, max={g.abs().max().item():.6f}")


In [106]:
import importlib, sys, time
import tgn.model.loss as loss_mod
importlib.reload(loss_mod)
longitudinal_modularity_loss = loss_mod.longitudinal_modularity_loss
NUM_EPOCHS = 30


for epoch in range(NUM_EPOCHS):
    tgn = tgn.train()
    start_epoch_time = time.time()
    if USE_MEMORY:
        tgn.memory.__init_memory__()
    state_mgr.reset_time_pos()
    state_mgr.reset_p_prev_uniform()
    tgn.set_neighbor_finder(ngh_finder)
    
    epoch_loss = 0.0
    epoch_obs_loss = 0.0
    epoch_null_loss = 0.0
    epoch_csc_loss = 0.0
    epoch_balance_loss = 0.0
    epoch_conf_loss = 0.0
    epoch_steps = 0

    logger.info(f'Starting epoch {epoch+1} / {NUM_EPOCHS} ')
    for k in range(0, num_batches):
        print(f'Processing batch {k+1} / {num_batches} ')
        start_idx = BATCH_SIZE * k
        end_idx = min(num_instance, BATCH_SIZE * (k + 1))

        sources_batch = full_data.sources[start_idx:end_idx]
        destinations_batch = full_data.destinations[start_idx:end_idx]
        timestamps_batch = full_data.timestamps[start_idx:end_idx]
        edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]
        
        src = torch.as_tensor(sources_batch, dtype=torch.long, device=device)
        dst = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)
        ts  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)

        delta_src, delta_dst, occ_counts = state_mgr.get_delta_for_batch(src, dst)

        p_src, p_dst = tgn.compute_community_prob(sources_batch, destinations_batch, timestamps_batch, edge_idxs_batch) # [B, K]
        p_src_old, p_dst_old = state_mgr.read_old_p_batch(
            start_idx, end_idx,
            device=device,
            dtype=p_src.dtype,
            non_blocking=False
        )
        
    
        if not torch.isfinite(p_src).all(): raise RuntimeError("p_src NaN right after forward")
        if not torch.isfinite(p_dst).all(): raise RuntimeError("p_dst NaN right after forward")

        delta_a_nodes, last_p_nodes = build_batch_updates(src, dst,
                                                          p_src, p_dst,
                                                          p_src_old, p_dst_old,
                                                          delta_src, delta_dst)
        p_nodes = torch.cat([p_src, p_dst])
        pi_batch = p_nodes.mean(dim=0)


        # Compute longitudinal modularity loss
        loss, last_components, terms_raw = longitudinal_modularity_loss(
            p_src, p_dst,
            src, dst,
            delta_a_nodes,
            state_mgr.A_old, state_mgr.S_old, state_mgr.global_degree,
            state_mgr.m, state_mgr.total_duration,
            state_mgr.p_prev,
            obs_weight=0.0,
            null_weight=100.0,
            csc_weight=0.0,
            csc_norm="l2"
        )
        obs_loss = terms_raw["obs"]
        null_loss = terms_raw["null"]

        dir_loss = tgn.dirichlet_prior(pi_batch)
        loss += 0 * dir_loss

        # 1) null 项梯度
        optimizer.zero_grad(set_to_none=True)
        null_loss.backward(retain_graph=True)
        print_grads("Null Loss")

        """
        # 2) obs 项梯度
        optimizer.zero_grad(set_to_none=True)
        obs_loss.backward(retain_graph=True)
        print_grads("Obs Loss")


        optimizer.zero_grad(set_to_none=True)
        dir_loss.backward(retain_graph=True)
        print_grads("Dirichlet Loss")
        """
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        
        state_mgr.commit_batch(delta_a_nodes, last_p_nodes, occ_counts, update_A_curr=False)
        if USE_MEMORY:
            tgn.memory.detach_memory()
        
        epoch_loss += float(loss.detach().item())
        epoch_obs_loss  += float(last_components[0].item())
        epoch_null_loss += float(last_components[1].item())
        epoch_csc_loss  += float(last_components[2].item())
        epoch_steps += 1
        # print(f'Epoch {epoch} Batch {k+1}/{num_batches} null model loss: {epoch_null_loss / epoch_steps} ')
    if epoch_steps > 0:
        mean_loss = epoch_loss / epoch_steps
        mean_obs  = epoch_obs_loss  / epoch_steps
        mean_null = epoch_null_loss / epoch_steps
        mean_csc  = epoch_csc_loss  / epoch_steps
    
        logger.info(
            f"[Epoch {epoch}] mean loss={mean_loss:.6f} | "
            f"obs={mean_obs:.6f}, null={mean_null:.6f}, csc={mean_csc:.6f}, dirichlet loss={dir_loss} "
        )

    # build null buffers
    tgn.eval()
    with torch.no_grad():
        if USE_MEMORY:
            tgn.memory.__init_memory__()
        state_mgr.reset_time_pos()
        state_mgr.reset_p_prev_uniform()
        state_mgr.reset_curr_from_zero()
        tgn.set_neighbor_finder(ngh_finder)
        for k in range(num_batches):
            start_idx = BATCH_SIZE * k
            end_idx = min(num_instance, BATCH_SIZE * (k + 1))

            sources_batch = full_data.sources[start_idx:end_idx]
            destinations_batch = full_data.destinations[start_idx:end_idx]
            timestamps_batch = full_data.timestamps[start_idx:end_idx]
            edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]
            
            src = torch.as_tensor(sources_batch, dtype=torch.long, device=device)
            dst = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)
            ts  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)

            delta_src, delta_dst, occ_counts = state_mgr.get_delta_for_batch(src, dst)

            p_src_ng, p_dst_ng = tgn.compute_community_prob(sources_batch, destinations_batch, timestamps_batch, edge_idxs_batch) # [B, K]

            state_mgr.write_old_p_batch(
                start_idx, end_idx,
                p_src_ng.detach(), p_dst_ng.detach()
            )

            delta_a_nodes, last_p_nodes = build_batch_updates_abs(  
                src, dst,
                p_src_ng, p_dst_ng,
                delta_src, delta_dst)
            state_mgr.commit_batch(delta_a_nodes, last_p_nodes, occ_counts, update_A_curr=True)
            
            if USE_MEMORY:
                tgn.memory.detach_memory()
        state_mgr.finalize_curr()
        state_mgr.promote_curr_to_old(copy_A=True)
    if USE_MEMORY:
        # Backup and restore memory
        memory_backup = tgn.memory.backup_memory()
        tgn.memory.restore_memory(memory_backup)

INFO:root:Starting epoch 1 / 30 


Processing batch 1 / 1 


INFO:root:[Epoch 0] mean loss=84.964767 | obs=-0.503088, null=0.849648, csc=0.028696, dirichlet loss=-0.5203986167907715 



=== Batch 1 Null Loss Gradients ===
time_encoder.w.weight: norm=0.001766, max=0.001466
time_encoder.w.bias: norm=0.000006, max=0.000003
embedding_module.attention_models.0.merger.fc1.weight: norm=0.000063, max=0.000014
embedding_module.attention_models.0.merger.fc1.bias: norm=0.000039, max=0.000024
embedding_module.attention_models.0.merger.fc2.weight: norm=0.000042, max=0.000012
embedding_module.attention_models.0.merger.fc2.bias: norm=0.000040, max=0.000019
embedding_module.attention_models.0.multi_head_target.q_proj_weight: norm=0.000006, max=0.000001
embedding_module.attention_models.0.multi_head_target.k_proj_weight: norm=0.000005, max=0.000002
embedding_module.attention_models.0.multi_head_target.v_proj_weight: norm=0.000088, max=0.000009
embedding_module.attention_models.0.multi_head_target.in_proj_bias: norm=0.000026, max=0.000009
embedding_module.attention_models.0.multi_head_target.out_proj.weight: norm=0.000114, max=0.000025
embedding_module.attention_models.0.multi_head_ta

INFO:root:Starting epoch 2 / 30 


...


=== Batch 1 Null Loss Gradients ===
time_encoder.w.weight: norm=0.065565, max=0.033273
time_encoder.w.bias: norm=0.000237, max=0.000098
embedding_module.attention_models.0.merger.fc1.weight: norm=0.007483, max=0.001208
embedding_module.attention_models.0.merger.fc1.bias: norm=0.001827, max=0.000923
embedding_module.attention_models.0.merger.fc2.weight: norm=0.006433, max=0.001886
embedding_module.attention_models.0.merger.fc2.bias: norm=0.001711, max=0.000639
embedding_module.attention_models.0.multi_head_target.q_proj_weight: norm=0.000249, max=0.000029
embedding_module.attention_models.0.multi_head_target.k_proj_weight: norm=0.000317, max=0.000097
embedding_module.attention_models.0.multi_head_target.v_proj_weight: norm=0.006658, max=0.000720
embedding_module.attention_models.0.multi_head_target.in_proj_bias: norm=0.002089, max=0.000720
embedding_module.attention_models.0.multi_head_target.out_proj.weight: norm=0.007848, max=0.001274
embedding_module.attention_models.0.multi_head_ta

INFO:root:Starting epoch 30 / 30 


Processing batch 1 / 1 


INFO:root:[Epoch 29] mean loss=81.489105 | obs=-0.848487, null=0.814891, csc=0.165956, dirichlet loss=1.1395502090454102 



=== Batch 1 Null Loss Gradients ===
time_encoder.w.weight: norm=0.047701, max=0.030487
time_encoder.w.bias: norm=0.000228, max=0.000136
embedding_module.attention_models.0.merger.fc1.weight: norm=0.006844, max=0.001198
embedding_module.attention_models.0.merger.fc1.bias: norm=0.001640, max=0.000858
embedding_module.attention_models.0.merger.fc2.weight: norm=0.005792, max=0.001576
embedding_module.attention_models.0.merger.fc2.bias: norm=0.001425, max=0.000554
embedding_module.attention_models.0.multi_head_target.q_proj_weight: norm=0.000304, max=0.000040
embedding_module.attention_models.0.multi_head_target.k_proj_weight: norm=0.000373, max=0.000128
embedding_module.attention_models.0.multi_head_target.v_proj_weight: norm=0.006186, max=0.000646
embedding_module.attention_models.0.multi_head_target.in_proj_bias: norm=0.001942, max=0.000646
embedding_module.attention_models.0.multi_head_target.out_proj.weight: norm=0.007236, max=0.001233
embedding_module.attention_models.0.multi_head_ta

In [93]:
print(any(p is tgn.dirichlet_prior.logits for g in optimizer.param_groups for p in g["params"]))
print(tgn.dirichlet_prior.logits.data[:5])

True
tensor([-0.0079,  0.0079], device='mps:0')


In [103]:
print(state_mgr.node_lifespans)

tensor([ 860.,  846.,  866.,  838.,  893.,  869.,  972., 1001.,  852.,  902.,
        1015.,  901.,  839.,  967., 1008.,  962., 1007.,  916., 1061.,  970.],
       device='mps:0')


In [93]:
print(state_mgr.global_degree)

tensor([16., 15., 14., 25., 23., 31., 16., 24., 15., 18., 21., 20., 12., 11.,
        18., 17., 11., 23., 15., 19.], device='mps:0')


In [69]:
print(state_mgr.A_old)

tensor([[221.4301, 639.5699],
        [201.4665, 645.5334],
        [ 22.8080, 844.1920],
        [247.1931, 591.8069],
        [292.0127, 601.9874],
        [297.9079, 572.0920],
        [381.1953, 591.8047],
        [325.2942, 676.7057],
        [300.2756, 552.7244],
        [385.7289, 517.2711],
        [354.3191, 608.6808],
        [415.9769, 600.0231],
        [347.8544, 554.1456],
        [163.3475, 676.6526],
        [ 34.1538, 973.8463],
        [264.3596, 703.6404],
        [439.8022, 569.1978],
        [421.5834, 495.4166],
        [459.5651, 602.4349],
        [359.3710, 611.6290]], device='mps:0')


In [94]:
tgn.eval()
with torch.no_grad():
    alpha = tgn.dirichlet_prior.alpha()          # [K]
    print("alpha:", alpha.detach().cpu())
    print("alpha_sum:", alpha.sum().item())
    print("logits:", tgn.dirichlet_prior.logits.detach().cpu())

alpha: tensor([2.4812, 2.5208])
alpha_sum: 5.001999855041504
logits: tensor([-0.0079,  0.0079])


In [109]:
print(state_mgr.S_old)

tensor([ 1807.8989, 10902.0215], device='mps:0')


In [ ]:
print((state_mgr.S_old.pow(2))/((2 * state_mgr.m) ** 2 * state_mgr.total_duration))

tensor([0.0227, 0.8252], device='mps:0')


In [98]:
print(state_mgr.m)

182.0


In [99]:
print(state_mgr.total_duration)

1087.0


In [23]:
print(node2id)

{np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3, np.int64(4): 4, np.int64(5): 5, np.int64(6): 6, np.int64(7): 7, np.int64(8): 8, np.int64(9): 9, np.int64(11): 10, np.int64(12): 11, np.int64(13): 12, np.int64(15): 13, np.int64(16): 14, np.int64(10): 15, np.int64(14): 16, np.int64(17): 17, np.int64(18): 18, np.int64(19): 19}


In [55]:
id2node = {idx: node for node, idx in node2id.items()}

In [56]:
print(id2node)

{0: np.int64(0), 1: np.int64(1), 2: np.int64(2), 3: np.int64(3), 4: np.int64(4), 5: np.int64(5), 6: np.int64(6), 7: np.int64(7), 8: np.int64(8), 9: np.int64(9), 10: np.int64(11), 11: np.int64(12), 12: np.int64(13), 13: np.int64(15), 14: np.int64(16), 15: np.int64(10), 16: np.int64(14), 17: np.int64(17), 18: np.int64(18), 19: np.int64(19)}


In [71]:
import json
import numpy as np
import pandas as pd
import torch

def export_probs_to_csv(
    tgn,
    full_data,
    BATCH_SIZE: int,
    out_csv_path: str,
    id2node: dict = None,   # 用这个：id -> 原始node
):
    tgn.eval()
    num_instance = len(full_data.sources)
    rows = []

    def _to_py_int(x):
        if isinstance(x, np.generic):
            return x.item()
        if torch.is_tensor(x):
            return x.item()
        return x

    def _map_id_to_node_name(x):
        x = _to_py_int(x)  # x 此时是 node_id
        if id2node is None:
            return int(x)  # 不提供就输出id
        return id2node[int(x)]  # 输出原始node（可能是int或str）

    with torch.no_grad():
        for k in range((num_instance + BATCH_SIZE - 1) // BATCH_SIZE):
            start_idx = BATCH_SIZE * k
            end_idx = min(num_instance, BATCH_SIZE * (k + 1))

            sources_batch = full_data.sources[start_idx:end_idx]
            destinations_batch = full_data.destinations[start_idx:end_idx]
            timestamps_batch = full_data.timestamps[start_idx:end_idx]
            edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]

            p_src, p_dst = tgn.compute_community_prob(
                sources_batch, destinations_batch, timestamps_batch, edge_idxs_batch
            )

            p_src_np = p_src.detach().cpu().float().numpy()
            p_dst_np = p_dst.detach().cpu().float().numpy()

            src_commu = p_src_np.argmax(axis=1).astype(np.int64)
            dst_commu = p_dst_np.argmax(axis=1).astype(np.int64)

            for i in range(end_idx - start_idx):
                rows.append({
                    "source": _map_id_to_node_name(sources_batch[i]),
                    "destination": _map_id_to_node_name(destinations_batch[i]),
                    "timestamp": float(_to_py_int(timestamps_batch[i])),
                    "p_src": json.dumps(p_src_np[i].tolist()),
                    "p_dst": json.dumps(p_dst_np[i].tolist()),
                    "source_commu": int(src_commu[i]),
                    "destination_commu": int(dst_commu[i]),
                })

    df = pd.DataFrame(rows, columns=[
        "source", "destination", "timestamp",
        "p_src", "p_dst", "source_commu", "destination_commu"
    ])
    df.to_csv(out_csv_path, index=False)
    print(f"Saved: {out_csv_path}  (rows={len(df)})")

export_probs_to_csv(tgn, full_data, BATCH_SIZE, "result/TGN_community.csv", id2node=id2node)

Saved: result/TGN_community.csv  (rows=182)
